# Task 1: Binary sentiment classification using IMDB data.

Task 1 focuses on binary sentiment classification, aiming to predict whether a movie review is positive or negative. The IMDB dataset is well-suited for this task because it is conveniently preprocessed in tf.keras.datasets.imdb, allowing for immediate use. Since the data is composed of sequential word inputs, it is naturally compatible with sequence models such as RNNs, LSTMs, and 1D CNNs. Moreover, the dataset is small enough to be trained efficiently without the need for heavy computational resources.

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1) Load IMDB dataset
vocab_size = 10000  # keep top 10k words
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# 2) Get word index (mapping int -> word)
word_index = imdb.get_word_index()
index_to_word = {idx + 3: word for word, idx in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

# 3) Utility: decode review back to text
def decode_review(sequence):
    return " ".join([index_to_word.get(i, "?") for i in sequence])

# 4) Visualize a few samples
for i in range(5):
    text = decode_review(x_train[i])
    label = "positive" if y_train[i] == 1 else "negative"
    num_words = len(x_train[i])
    print(f"Sample {i+1}")
    print(f"Label: {label}")
    print(f"Text: {text[:300]}...")  # truncate for readability
    print(f"Num words: {num_words}")
    print("-" * 80)


Sample 1
Label: positive
Text: <START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the...
Num words: 218
--------------------------------------------------------------------------------
Sample 2
Label: negative
Text: <START> big hair big boobs bad music and a giant safety pin these are the words to best describe this terrible movie i love cheesy horror movies and i've seen hundreds but this had got to be on of the worst ever made the plot is paper thin and ridiculous the acting is an abomination the script is co...
Num words: 189
--------------------------------------------------------------------------------
Sample 3
Label: negative
Text: <START> this has to be one of the worst films of the 1990s when my friends i were watching this film being 

# Implementation using 1D CNN

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)

# 1) Data
vocab_size = 20_000
max_len    = 256          # sequence length
embed_dim  = 16           # feature size per timestep (kept same as LSTM/RNN demos)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

# 2) Model: 2-layer CNN1D (Conv -> Pool) x2, then Flatten -> Dense(1)
inputs = Input(shape=(max_len,), name="inputs")
x = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x = Conv1D(32, 5, padding="same", activation="relu")(x)
x = MaxPooling1D(2)(x)

x = Conv1D(64, 5, padding="same", activation="relu")(x)
x = MaxPooling1D(2)(x)

x = Flatten(name="flatten")(x)
x = Dense(64, activation="relu")(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# 3) Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,            # you can set to 10 if you prefer
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

# 4) Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Embedding)               │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 256, 32)        │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 595,169 (2.27 MB)

 Trainable params: 595,169 (2.27 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
157/157 - 2s - 13ms/step - accuracy: 0.7099 - loss: 0.5105 - val_accuracy: 0.8836 - val_loss: 0.2885
Epoch 2/5
157/157 - 1s - 7ms/step - accuracy: 0.9190 - loss: 0.2050 - val_accuracy: 0.8764 - val_loss: 0.3262
Epoch 3/5
157/157 - 1s - 8ms/step - accuracy: 0.9488 - loss: 0.1354 - val_accuracy: 0.8774 - val_loss: 0.3219
Test accuracy: 0.8745


# Exercise 1a (LSTM):
Build an LSTM model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.


In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)

vocab_size = 20_000
max_len    = 256
embed_dim  = 16

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

# Embedding -> LSTM -> Dense(1)
lstm_units = 254

inputs  = Input(shape=(max_len,), name="inputs")
x       = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x       = LSTM(lstm_units, name="lstm")(x)
outputs = Dense(1, activation="sigmoid", name="out")(x)

model_lstm = Model(inputs=inputs, outputs=outputs)
model_lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_lstm.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history_lstm = model_lstm.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

test_loss_lstm, test_acc_lstm = model_lstm.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc_lstm:.4f}")
print(f"Total params : {model_lstm.count_params():,}   (1D CNN demo: 595,169)")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Embedding)               │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 254)            │       275,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (Dense)                     │ (None, 1)              │           255 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 595,591 (2.27 MB)

 Trainable params: 595,591 (2.27 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
157/157 - 84s - 534ms/step - accuracy: 0.7135 - loss: 0.5548 - val_accuracy: 0.8486 - val_loss: 0.3555
Epoch 2/5
157/157 - 126s - 803ms/step - accuracy: 0.8864 - loss: 0.2834 - val_accuracy: 0.8714 - val_loss: 0.3211
Epoch 3/5
157/157 - 99s - 632ms/step - accuracy: 0.9169 - loss: 0.2239 - val_accuracy: 0.8354 - val_loss: 0.3843
Epoch 4/5
157/157 - 114s - 729ms/step - accuracy: 0.9403 - loss: 0.1693 - val_accuracy: 0.8218 - val_loss: 0.4533
Test accuracy: 0.8644
Total params : 595,591   (1D CNN demo: 595,169)


# Exercise 1b (RNN):
Build an RNN model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)

vocab_size = 20_000
max_len    = 256
embed_dim  = 16

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

rnn_units = 512

inputs  = Input(shape=(max_len,), name="inputs")
x       = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x       = SimpleRNN(rnn_units, name="rnn")(x)
outputs = Dense(1, activation="sigmoid", name="out")(x)

model_rnn = Model(inputs=inputs, outputs=outputs)
model_rnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_rnn.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history_rnn = model_rnn.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

test_loss_rnn, test_acc_rnn = model_rnn.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc_rnn:.4f}")
print(f"Total params : {model_rnn.count_params():,}   (1D CNN demo: 595,169)")


### Parameter-matched comparison: 1D CNN vs LSTM vs SimpleRNN

All three models share the same input pipeline (`vocab_size=20,000`, `max_len=256`,
`embed_dim=16`), so the 320,000-parameter Embedding is common to all of them. Only the
"body" of the network differs, which is why each recurrent model's hidden size is chosen
to make the **total** parameter count land close to the CNN's 595,169.

- **LSTM** has 4 gates, so its cost grows as `4u^2` -> needs **u = 254**
- **SimpleRNN** has a single hidden state, costing `u^2` -> needs **u = 512**
  (roughly 2x the LSTM's units, since `4u_lstm^2 = u_rnn^2` implies `u_rnn = 2 * u_lstm`)


In [ ]:
cnn_loss, cnn_acc = model.evaluate(x_test, y_test, verbose=0)
cnn_params = model.count_params()

rows = [
    ("1D CNN (demo)",    cnn_params,                 cnn_acc),
    ("LSTM (Ex 1a)",     model_lstm.count_params(),  test_acc_lstm),
    ("SimpleRNN (Ex 1b)", model_rnn.count_params(),  test_acc_rnn),
]

print(f"{'Model':<20}{'Total params':>14}{'vs CNN':>10}{'Test acc':>11}")
print("-" * 55)
for name, p, acc in rows:
    print(f"{name:<20}{p:>14,}{(p / cnn_params - 1) * 100:>9.2f}%{acc:>11.4f}")

print("\nBest validation accuracy per epoch:")
for name, hist in (("1D CNN", history), ("LSTM", history_lstm), ("SimpleRNN", history_rnn)):
    print(f"  {name:<12} {max(hist.history['val_accuracy']):.4f}  "
          f"({len(hist.history['val_accuracy'])} epochs run)")


---


# Task 2: Reuters Newswire Topics (multi-class classification with 46 labels).

Task 2 involves predicting the topic category of a short newswire article among 46 possible classes. The Reuters dataset is conveniently available in tf.keras.datasets.reuters, so you can load and preprocess it immediately. Its sequences are typically shorter than those in IMDB, which makes training faster and well suited to classroom demos or quick iterations. Because it is a multi-class problem with compact inputs, Reuters is an excellent benchmark for comparing different model architectures—such as 1D CNNs, RNNs/LSTMs, and Transformers—on the same text-classification task.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import reuters

# 1) Load Reuters dataset
vocab_size = 10000   # keep top 10k words
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=vocab_size)

# 2) Get word index mapping (int -> word)
word_index = reuters.get_word_index()
index_to_word = {idx + 3: word for word, idx in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

# 3) Reuters topic labels (from Reuters-21578 dataset, 46 classes)
reuters_topics = {
    0: "cocoa", 1: "grain", 2: "veg-oil", 3: "earn", 4: "acq",
    5: "wheat", 6: "corn", 7: "crude", 8: "money-fx", 9: "interest",
    10: "ship", 11: "trade", 12: "reserves", 13: "cotton", 14: "coffee",
    15: "sugar", 16: "gold", 17: "tin", 18: "strategic-metal", 19: "livestock",
    20: "retail", 21: "ipi", 22: "iron-steel", 23: "rubber", 24: "heat",
    25: "jobs", 26: "lei", 27: "money-supply", 28: "alum", 29: "oilseed",
    30: "gas", 31: "cpi", 32: "money-market", 33: "palm-oil", 34: "dmk-mark",
    35: "bop", 36: "gnp", 37: "silver", 38: "zinc", 39: "income",
    40: "lead", 41: "housing", 42: "copper", 43: "meal-feed", 44: "ipi-indicator",
    45: "strategic"
}

# 4) Utility: decode newswire back to text
def decode_newswire(sequence):
    return " ".join([index_to_word.get(i, "?") for i in sequence])

# 5) Visualize a few samples
for i in range(5):
    text = decode_newswire(x_train[i])
    label_id = y_train[i]
    label_name = reuters_topics.get(label_id, "unknown")
    num_words = len(x_train[i])
    print(f"Sample {i+1}")
    print(f"Label: {label_id} ({label_name})")
    print(f"Num words: {num_words}")
    print(f"Text: {text[:300]}...")  # truncate for readability
    print("*" * 80)


# Implementation using 1D CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
import numpy as np

tf.random.set_seed(42)

# 1) Data (Reuters)
vocab_size = 20_000         # keep top words
max_len    = 256            # sequence length
embed_dim  = 16             # features per timestep

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.reuters.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

num_classes = int(max(y_train.max(), y_test.max()) + 1)
y_train = to_categorical(y_train, num_classes)
y_test  = to_categorical(y_test,  num_classes)

# 2) Model: 2-layer CNN1D (Conv -> Pool) x2, then Flatten -> Dense(num_classes)
inputs = Input(shape=(max_len,), name="inputs")
x = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x = Conv1D(32, 5, padding="same", activation="relu")(x)
x = MaxPooling1D(2)(x)

x = Conv1D(64, 5, padding="same", activation="relu")(x)
x = MaxPooling1D(2)(x)

x = Flatten(name="flatten")(x)
x = Dense(64, activation="relu")(x)
outputs = Dense(num_classes, activation="softmax", name="out")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# 3) Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,                  # adjust as you like
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

# 4) Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")


# Exercise 2a (LSTM) (Optional):
Build an LSTM model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)

vocab_size = 20_000
max_len    = 256
embed_dim  = 16

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.reuters.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

num_classes = int(max(y_train.max(), y_test.max()) + 1)
y_train = to_categorical(y_train, num_classes)
y_test  = to_categorical(y_test,  num_classes)

#    Parameter budget: the 1D CNN demo totals 598,094 params, of which the Embedding
#    accounts for 20,000 x 16 = 320,000, leaving 278,094 for the model body.
#    An LSTM layer costs 4 * ((embed_dim + units) * units + units) (4 gates), and the
#    softmax layer over 46 topics adds num_classes * units + num_classes. Solving for
#    that budget gives units ~= 250.
lstm_units = 250

inputs  = Input(shape=(max_len,), name="inputs")
x       = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x       = LSTM(lstm_units, name="lstm")(x)
outputs = Dense(num_classes, activation="softmax", name="out")(x)

model_r_lstm = Model(inputs=inputs, outputs=outputs)
model_r_lstm.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model_r_lstm.summary()


callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history_r_lstm = model_r_lstm.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

test_loss_r_lstm, test_acc_r_lstm = model_r_lstm.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc_r_lstm:.4f}")
print(f"Total params : {model_r_lstm.count_params():,}   (1D CNN demo: 598,094)")


# Exercise 2b (RNN)  (Optional):
Build an RNN model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)

vocab_size = 20_000
max_len    = 256
embed_dim  = 16

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.reuters.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

num_classes = int(max(y_train.max(), y_test.max()) + 1)
y_train = to_categorical(y_train, num_classes)
y_test  = to_categorical(y_test,  num_classes)

#    A SimpleRNN layer costs only (embed_dim + units) * units + units - a single hidden
#    state, no gates. Matching the same 278,094-parameter body budget therefore needs
#    roughly twice the units of the LSTM: units ~= 497.
rnn_units = 497

inputs  = Input(shape=(max_len,), name="inputs")
x       = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x       = SimpleRNN(rnn_units, name="rnn")(x)
outputs = Dense(num_classes, activation="softmax", name="out")(x)

model_r_rnn = Model(inputs=inputs, outputs=outputs)
model_r_rnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model_r_rnn.summary()


callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history_r_rnn = model_r_rnn.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

test_loss_r_rnn, test_acc_r_rnn = model_r_rnn.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc_r_rnn:.4f}")
print(f"Total params : {model_r_rnn.count_params():,}   (1D CNN demo: 598,094)")


### Parameter-matched comparison: Reuters 1D CNN vs LSTM vs SimpleRNN

Same idea as Task 1, but the output layer now has 46 units instead of 1. All three models
share `vocab_size=20,000`, `max_len=256` and `embed_dim=16`, so the 320,000-parameter
Embedding is common to all of them and only the body of the network differs.

The 1D CNN demo totals 598,094 params, leaving **278,094** for the body once the Embedding
is subtracted. Matching that budget:

- **LSTM** costs `4 * ((16 + u) * u + u)` plus the softmax layer `46u + 46` -> **u = 250**
- **SimpleRNN** costs `((16 + u) * u + u)` plus the same softmax layer -> **u = 497**

The same relationship as Task 1 shows up again: the SimpleRNN needs roughly **twice** the
hidden units of the LSTM, because the LSTM pays for 4 gates (`4u^2`) while the SimpleRNN
has a single hidden state (`u^2`).


In [ ]:
cnn_loss_r, cnn_acc_r = model.evaluate(x_test, y_test, verbose=0)
cnn_params_r = model.count_params()

rows = [
    ("1D CNN (demo)",     cnn_params_r,                 cnn_acc_r),
    ("LSTM (Ex 2a)",      model_r_lstm.count_params(),  test_acc_r_lstm),
    ("SimpleRNN (Ex 2b)", model_r_rnn.count_params(),   test_acc_r_rnn),
]

print(f"{'Model':<20}{'Total params':>14}{'vs CNN':>10}{'Test acc':>11}")
print("-" * 55)
for name, p, acc in rows:
    print(f"{name:<20}{p:>14,}{(p / cnn_params_r - 1) * 100:>9.2f}%{acc:>11.4f}")

print("\nBest validation accuracy per epoch:")
for name, hist in (("1D CNN", history), ("LSTM", history_r_lstm), ("SimpleRNN", history_r_rnn)):
    print(f"  {name:<12} {max(hist.history['val_accuracy']):.4f}  "
          f"({len(hist.history['val_accuracy'])} epochs run)")


---


# Task 3: Shakespeare / Tiny Shakespeare Character Dataset (character-level language modeling).
Task 3 focuses on next-character prediction, where the model learns to generate text one character at a time by predicting the next character in a sequence. The Tiny Shakespeare dataset is a classic benchmark for demonstrating RNNs and LSTMs because it is small (less than 1 MB), fun to work with. It provides a simple yet effective way to showcase how sequence models can capture language patterns and generate coherent text continuations.

# Visualization of text samples

In [ ]:
import tensorflow as tf
import io

# 1) Download Tiny Shakespeare text
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = tf.keras.utils.get_file("tiny_shakespeare.txt", origin=url)
text = io.open(path, encoding="utf-8").read()

print("Total characters in corpus:", len(text))

# 2) Build char vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Unique characters (vocab size):", vocab_size)
print("Character set:", chars)

# 3) Map char <-> int
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

# 4) Encode the whole text
encoded = [stoi[c] for c in text]

# 5) Visualize a few samples
print("\n=== Sample visualization ===")
for i in range(3):
    snippet = text[i*200:(i+1)*200]   # take 200-char snippets
    encoded_snippet = encoded[i*200:(i+1)*200]
    print(f"\nSample {i+1}")
    print("Raw text:\n", snippet[:300].replace("\n", "\\n"))  # replace newline for clarity
    print("Encoded IDs:\n", encoded_snippet[:50], "...")      # show first 50 IDs
    print("*" * 80)


# Exercise 3a (LSTM) (Optional):
Build an LSTM model for next-character prediction.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)

seq_len = 60
step    = 3

sentences  = []
next_chars = []
for i in range(0, len(encoded) - seq_len, step):
    sentences.append(encoded[i : i + seq_len])
    next_chars.append(encoded[i + seq_len])

x_seq = np.array(sentences,  dtype="int32")
y_seq = np.array(next_chars, dtype="int32")

split = int(len(x_seq) * 0.9)
x_tr, y_tr = x_seq[:split], y_seq[:split]
x_te, y_te = x_seq[split:], y_seq[split:]

print(f"Training sequences : {x_tr.shape}")
print(f"Test sequences     : {x_te.shape}")
print(f"Characters (classes): {vocab_size}")
# step=3 yields ~370k overlapping windows.


# Embedding -> LSTM -> Dense(vocab_size, softmax)
#    Targets are integer character ids, so we use sparse_categorical_crossentropy
#    rather than one-hot encoding 370k x 65 arrays.
embed_dim  = 32
lstm_units = 128

inputs  = Input(shape=(seq_len,), name="inputs")
x       = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x       = LSTM(lstm_units, name="lstm")(x)
outputs = Dense(vocab_size, activation="softmax", name="out")(x)

model_c_lstm = Model(inputs=inputs, outputs=outputs)
model_c_lstm.compile(optimizer="adam",
                     loss="sparse_categorical_crossentropy",
                     metrics=["accuracy"])
model_c_lstm.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history_c_lstm = model_c_lstm.fit(
    x_tr, y_tr,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

test_loss_c_lstm, test_acc_c_lstm = model_c_lstm.evaluate(x_te, y_te, verbose=0)
print(f"Next-character accuracy on held-out text: {test_acc_c_lstm:.4f}")
print(f"Total params: {model_c_lstm.count_params():,}")


# Exercise 3b (RNN) (Optional):
Build an RNN model for next-character prediction.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)

# Data - reuses x_tr / y_tr / x_te / y_te built in Exercise 3a above.


embed_dim  = 32
rnn_units  = 128

inputs  = Input(shape=(seq_len,), name="inputs")
x       = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x       = SimpleRNN(rnn_units, name="rnn")(x)
outputs = Dense(vocab_size, activation="softmax", name="out")(x)

model_c_rnn = Model(inputs=inputs, outputs=outputs)
model_c_rnn.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])
model_c_rnn.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history_c_rnn = model_c_rnn.fit(
    x_tr, y_tr,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

test_loss_c_rnn, test_acc_c_rnn = model_c_rnn.evaluate(x_te, y_te, verbose=0)
print(f"Next-character accuracy on held-out text: {test_acc_c_rnn:.4f}")
print(f"Total params: {model_c_rnn.count_params():,}")


### Character-level language modelling: LSTM vs SimpleRNN

Task 3 is a different problem from Tasks 1 and 2. Instead of mapping one whole sequence to
a single label, the model now predicts the **next character** at every position. The output
layer therefore has one unit per character (65 of them here), and the loss is
`sparse_categorical_crossentropy` over integer character ids.


- The sliding-window scheme follows `8.1-text-generation-with-lstm.ipynb`: a
  window of 60 characters predicts the 61st, and a new window starts every 3 characters.
  That turns a 1.1M-character corpus into roughly 370k overlapping training pairs.
- The pairs are split **chronologically** - first 90% for training, last 10% for testing -
  rather than at random, so the held-out set is genuinely unseen text rather than
  fragments of sentences the model already memorised.


In [ ]:
def generate_text(model, seed_text, n_chars=300, temperature=0.8):
    generated = seed_text
    for _ in range(n_chars):
        window = [stoi[c] for c in generated[-seq_len:]]
        window = tf.constant([window], dtype=tf.int32)
        preds  = model(window, training=False).numpy()[0].astype("float64")
        # temperature re-weighting: <1 sharpens the distribution, >1 flattens it
        preds  = np.log(preds + 1e-8) / temperature
        preds  = np.exp(preds) / np.sum(np.exp(preds))
        generated += itos[np.random.choice(len(preds), p=preds)]
    return generated

seed = text[:seq_len].replace("\n", "\\n")
print(f"Seed: {seed!r}\n")

for name, m in (("LSTM", model_c_lstm), ("SimpleRNN", model_c_rnn)):
    print("=" * 78)
    print(f"{name} (temperature=0.8)")
    print("=" * 78)
    print(generate_text(m, text[:seq_len], n_chars=300, temperature=0.8))
    print()


for name, m in (("LSTM", model_c_lstm), ("SimpleRNN", model_c_rnn)):
    print("=" * 78)
    print(f"{name} (temperature=0.3)")
    print("=" * 78)
    print(generate_text(m, text[:seq_len], n_chars=300, temperature=0.3))
    print()
